# 05. Dashboarding

Esta notebook documenta la etapa final del flujo curado: como pasar del dataset reducido y limpio a archivos auxiliares listos para construir un dashboard. En esta version del proyecto, el dashboard final se arma fuera de Python, pero la preparacion de datos y la trazabilidad del proceso quedan reproducibles aqui.

## Alcance

- validar que la version reducida soporte las vistas del dashboard
- ejecutar el pipeline de generacion de assets
- revisar los archivos auxiliares producidos
- documentar el mapeo entre tabs, paneles y fuentes de datos
- dejar evidencia de los exports de dashboard disponibles en el proyecto


## Subpaso 1. Configuracion de rutas y artefactos

**Proposito:** conectar la notebook con el dataset reducido, el script de assets y los archivos exportados del dashboard.

**Resultado esperado:** acceso a `survey_data_cleaned_reduced.csv`, `create_dashboard_assets.py`, `data/dashboard_assets` y los PDFs ya exportados en `assets/`.


In [ ]:
from pathlib import Path
import subprocess
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
REDUCED_DATASET_PATH = DATA_DIR / "cleaned_outputs" / "survey_data_cleaned_reduced.csv"
SCRIPT_PATH = PROJECT_ROOT / "scripts" / "create_dashboard_assets.py"
DASHBOARD_ASSETS_DIR = DATA_DIR / "dashboard_assets"
DASHBOARD_CURRENT_PDF = PROJECT_ROOT / "assets" / "dashboard_current.pdf"
DASHBOARD_PROJECT_PDF = PROJECT_ROOT / "assets" / "dashboard_project.pdf"

REDUCED_DATASET_PATH, SCRIPT_PATH, DASHBOARD_ASSETS_DIR


## Subpaso 2. Validacion de columnas requeridas

**Proposito:** asegurar que la version reducida conserve todo lo necesario para las vistas del dashboard.

**Resultado:** se verifica que el dataset reducido todavia incluya columnas de tecnologias, edad, pais y nivel educativo.

**Hallazgo:** el recorte agresivo hecho en limpieza sigue siendo compatible con dashboarding porque preserva exactamente los bloques que alimentan tendencias tecnologicas y demografia.


In [ ]:
required_dashboard_columns = [
    "Age",
    "Country",
    "EdLevel",
    "LanguageHaveWorkedWith",
    "LanguageWantToWorkWith",
    "DatabaseHaveWorkedWith",
    "DatabaseWantToWorkWith",
    "PlatformHaveWorkedWith",
    "PlatformWantToWorkWith",
    "WebframeHaveWorkedWith",
    "WebframeWantToWorkWith",
]

reduced_df = pd.read_csv(REDUCED_DATASET_PATH, nrows=5)
column_check = pd.DataFrame(
    {
        "column_name": required_dashboard_columns,
        "available_in_reduced_dataset": [column in reduced_df.columns for column in required_dashboard_columns],
    }
)
column_check


## Subpaso 3. Generacion reproducible de assets del dashboard

**Proposito:** convertir el dataset reducido en archivos auxiliares listos para un dashboard de tres tabs.

**Resultado:** el script produce tablas Top 10 de tecnologias, tablas agrupadas para tabs y agregados demograficos.

**Hallazgo operativo:** externalizar esta logica en un script permite regenerar los assets sin depender del dashboarding tool ni rehacer transformaciones manuales.


In [ ]:
result = subprocess.run(
    ["python", str(SCRIPT_PATH)],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=True,
)

print(result.stdout)


## Subpaso 4. Revision del manifest y los assets generados

**Proposito:** comprobar que los archivos producidos cubren las necesidades del dashboard.

**Resultado:** el manifest resume que se generaron assets para uso actual, tendencias futuras y demografia.

**Hallazgos:** el pipeline entrega tanto archivos por grafico como tablas agrupadas, lo que da flexibilidad para herramientas como Looker Studio, Cognos o incluso un dashboard custom.


In [ ]:
manifest_path = DASHBOARD_ASSETS_DIR / "looker_assets_manifest.csv"
manifest_df = pd.read_csv(manifest_path)
manifest_df


## Subpaso 5. Mapeo entre tabs, paneles y fuentes de datos

**Proposito:** dejar documentado como se conectan los archivos auxiliares con la estructura del dashboard.

**Resultado:** se define un mapeo simple de tres tabs: `Current Technology Usage`, `Future Technology Trends` y `Demographics`.

**Hallazgo:** la historia del dashboard queda alineada con el resto del proyecto: primero estado actual, luego interes futuro y finalmente contexto demografico.


In [ ]:
dashboard_page_mapping = pd.DataFrame(
    [
        {"tab": "Current Technology Usage", "panel": "Top 10 Languages Used", "source_file": "current_top10_languages_used.csv", "metric": "count"},
        {"tab": "Current Technology Usage", "panel": "Top 10 Databases Used", "source_file": "current_top10_databases_used.csv", "metric": "count"},
        {"tab": "Current Technology Usage", "panel": "Top 10 Platforms Used", "source_file": "current_top10_platforms_used.csv", "metric": "count"},
        {"tab": "Current Technology Usage", "panel": "Top 10 Web Frameworks Used", "source_file": "current_top10_webframeworks_used_bubble.csv", "metric": "count"},
        {"tab": "Future Technology Trends", "panel": "Top 10 Languages Desired Next Year", "source_file": "future_top10_languages_desired.csv", "metric": "count"},
        {"tab": "Future Technology Trends", "panel": "Top 10 Databases Desired Next Year", "source_file": "future_top10_databases_desired.csv", "metric": "count"},
        {"tab": "Future Technology Trends", "panel": "Top 10 Desired Platforms", "source_file": "future_top10_platforms_desired.csv", "metric": "count"},
        {"tab": "Future Technology Trends", "panel": "Top 10 Desired Web Frameworks", "source_file": "future_top10_webframeworks_desired_bubble.csv", "metric": "count"},
        {"tab": "Demographics", "panel": "Respondents by Age", "source_file": "demographics_respondents_by_age.csv", "metric": "respondent_count"},
        {"tab": "Demographics", "panel": "Respondent Count by Country", "source_file": "demographics_respondent_count_by_country.csv", "metric": "respondent_count"},
        {"tab": "Demographics", "panel": "Distribution by Education Level", "source_file": "demographics_distribution_by_education_level.csv", "metric": "respondent_count"},
        {"tab": "Demographics", "panel": "Age by Education Level", "source_file": "demographics_respondent_count_by_age_and_education.csv", "metric": "respondent_count"},
    ]
)

dashboard_page_mapping.to_csv(DASHBOARD_ASSETS_DIR / "dashboard_page_mapping.csv", index=False)
dashboard_page_mapping


## Subpaso 6. Evidencia del dashboard exportado

**Proposito:** conectar el pipeline de assets con los entregables visuales ya exportados.

**Resultado:** la notebook referencia los PDFs disponibles como evidencia del dashboard final.

**Hallazgo operativo:** aunque el dashboard interactivo no se reconstruye dentro del repo, si queda claramente trazado que sus insumos y sus exports pertenecen al mismo flujo curado.


In [ ]:
dashboard_exports = pd.DataFrame(
    [
        {"artifact": "dashboard_current.pdf", "exists": DASHBOARD_CURRENT_PDF.exists(), "path": str(DASHBOARD_CURRENT_PDF)},
        {"artifact": "dashboard_project.pdf", "exists": DASHBOARD_PROJECT_PDF.exists(), "path": str(DASHBOARD_PROJECT_PDF)},
    ]
)
dashboard_exports


## Cierre de la etapa

Esta notebook cierra el flujo curado del proyecto: desde la version reducida del survey se generan assets trazables para el dashboard, se documenta su organizacion por tabs y se enlazan los exports finales disponibles. Con esto, el repositorio ya tiene una narrativa completa desde adquisicion hasta dashboarding.
